# Week 2: Contextual Data Fusion & Feature Engineering



In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import LabelEncoder
from scipy import stats

np.random.seed(42)
print("Random seed set to 42 for reproducibility.")

In [ ]:
# ---- CONFIGURATION: update these to match your Week 1 files ----
DATA_PATH = "ai4i2020.csv"     # your Week 1 cleaned dataset
DICT_PATH = "ai4i2020.csv"  # your Week 1 data dictionary

df = pd.read_csv(DATA_PATH)
data_dict = pd.read_csv(DICT_PATH)

print(df.shape)
df.head()

## Step 1 — Simulate Timestamps for the Telemetry

The AI4I dataset has no native timestamps — each row (`UDI`) is an independent sensor reading. To perform genuine time-based fusion (rather than a naive row-index join), we assign each reading a synthetic timestamp, treating the dataset as a continuous stream of readings taken at a fixed sampling interval (e.g., every 10 minutes), starting from an arbitrary plant start date.

This is a standard, defensible simulation choice for a synthetic tabular dataset that lacks native timestamps — 

**state this assumption explicitly in the report.**

In [ ]:
SAMPLING_INTERVAL_MINUTES = 10
START_DATE = datetime(2024, 1, 1, 0, 0, 0)

df = df.sort_values("UDI").reset_index(drop=True)
df["timestamp"] = [START_DATE + timedelta(minutes=SAMPLING_INTERVAL_MINUTES * i) for i in range(len(df))]

print(df["timestamp"].min(), "->", df["timestamp"].max())
df[["UDI", "timestamp"]].head()

## Step 2 — Simulate External Context Data

We simulate two independent external sources, each on its *own* time granularity (this granularity mismatch is intentional — it's what makes the merge in 

- **Ambient temperature** (hourly): daily sinusoidal cycle (cooler at night, warmer mid-afternoon) + slow seasonal drift + Gaussian noise. This is deliberately a *different* signal from the dataset's own `Air temperature [K]`, which is the machine's internal sensor reading.
- **Load density** (hourly, shift-based): weekly pattern (lower on weekends) + shift-based step changes (night/day/evening) + noise, representing plant-wide production load.

Because these are simulated at coarser granularity than the per-reading telemetry, Step 3 needs a nearest-timestamp merge rather than an exact join — mirroring a real-world fusion problem.

In [ ]:
def simulate_ambient_temperature(start, end, freq="h"):
    idx = pd.date_range(start, end, freq=freq)
    hours = idx.hour + idx.minute / 60
    daily_cycle = 5 * np.sin((hours - 9) / 24 * 2 * np.pi)   # peak mid-afternoon
    seasonal = 3 * np.sin((idx.dayofyear / 365) * 2 * np.pi)
    noise = np.random.normal(0, 0.8, size=len(idx))
    ambient_c = 22 + daily_cycle + seasonal + noise           # base 22 degC
    return pd.DataFrame({"timestamp": idx, "ambient_temp_C": ambient_c})

ambient_df = simulate_ambient_temperature(df["timestamp"].min().floor("h"),
                                           df["timestamp"].max().ceil("h"))
ambient_df.head()

In [ ]:
def simulate_load_density(start, end, freq="h"):
    idx = pd.date_range(start, end, freq=freq)
    is_weekend = idx.dayofweek >= 5
    shift = pd.cut(idx.hour, bins=[-1, 7, 15, 23], labels=["night", "day", "evening"])
    shift_base = shift.map({"night": 35, "day": 80, "evening": 60}).astype(float)
    weekend_drop = np.where(is_weekend, -25, 0)
    noise = np.random.normal(0, 4, size=len(idx))
    load_pct = np.clip(shift_base + weekend_drop + noise, 0, 100)
    return pd.DataFrame({"timestamp": idx, "load_density_pct": load_pct})

load_df = simulate_load_density(df["timestamp"].min().floor("h"),
                                 df["timestamp"].max().ceil("h"))
load_df.head()

## Step 3 — Fuse via Timestamp-Based Merge

We use `pd.merge_asof` to attach each telemetry reading to the *nearest* external reading in time. This mirrors a real-world data fusion scenario where sensor streams arrive at different sampling rates and must be aligned, not naively concatenated.

In [ ]:
df = df.sort_values("timestamp")
ambient_df = ambient_df.sort_values("timestamp")
load_df = load_df.sort_values("timestamp")

fused = pd.merge_asof(df, ambient_df, on="timestamp", direction="nearest")
fused = pd.merge_asof(fused, load_df, on="timestamp", direction="nearest")

fused[["UDI", "timestamp", "Air temperature [K]", "ambient_temp_C", "load_density_pct"]].head()

## Step 4 — Feature Engineering from Fused Data

We derive features that capture the *interaction* between internal sensor state and external context — this is where genuine predictive value, if any, would come from:

- `temp_differential_K`: machine air temperature vs. ambient temperature (a large gap may indicate poor cooling under load)
- `load_adjusted_torque`: torque normalized by plant load (the same torque means different things at 30% vs 90% load)
- `wear_per_load`: tool wear scaled by load density
- `ambient_temp_K`: ambient temp converted to Kelvin so it's on the same scale as the dataset's existing temperature columns

In [ ]:
fused["ambient_temp_K"] = fused["ambient_temp_C"] + 273.15
fused["temp_differential_K"] = fused["Air temperature [K]"] - fused["ambient_temp_K"]
fused["load_adjusted_torque"] = fused["Torque [Nm]"] / (fused["load_density_pct"] / 100 + 0.01)
fused["wear_per_load"] = fused["Tool wear [min]"] * (fused["load_density_pct"] / 100)

new_features = ["ambient_temp_K", "load_density_pct", "temp_differential_K",
                 "load_adjusted_torque", "wear_per_load"]
fused[new_features].describe()

## Step 5 — Update the Data Dictionary

Append entries for every new column so the dictionary stays the single source of truth for the project.

In [ ]:
new_entries = pd.DataFrame([
    {"feature": "timestamp", "type": "datetime", "source": "simulated",
     "description": f"Synthetic timestamp, {SAMPLING_INTERVAL_MINUTES}-min sampling interval from {START_DATE.date()}."},
    {"feature": "ambient_temp_C", "type": "float", "source": "simulated (external)",
     "description": "Simulated hourly ambient temperature (C): daily + seasonal cycle + noise."},
    {"feature": "ambient_temp_K", "type": "float", "source": "derived",
     "description": "ambient_temp_C converted to Kelvin."},
    {"feature": "load_density_pct", "type": "float", "source": "simulated (external)",
     "description": "Simulated hourly plant load (%): shift pattern + weekend drop + noise."},
    {"feature": "temp_differential_K", "type": "float", "source": "derived",
     "description": "Air temperature [K] minus ambient_temp_K."},
    {"feature": "load_adjusted_torque", "type": "float", "source": "derived",
     "description": "Torque [Nm] normalized by load_density_pct."},
    {"feature": "wear_per_load", "type": "float", "source": "derived",
     "description": "Tool wear [min] scaled by load_density_pct."},
])

data_dict = pd.concat([data_dict, new_entries], ignore_index=True)
data_dict.tail(7)

## Step 6 — Ablation Study: Does External Context Actually Help?

We compare two identical Random Forest classifiers predicting `Machine failure`, differing only in feature set:

- **Model A (baseline)**: original internal sensor features only
- **Model B (enriched)**: Model A's features + the 5 external/derived features from Step 4

We evaluate both with **repeated stratified 5-fold CV** (5 folds x 5 repeats = 25 scores per model) on ROC-AUC and F1, using the *same* fold splits for both models (paired design). We then run a **paired t-test** and the non-parametric **Wilcoxon signed-rank test** (robustness check, since CV fold scores aren't strictly independent) on the per-fold score differences to test whether the enriched model's improvement is statistically significant.

In [ ]:
le = LabelEncoder()
fused["Type_enc"] = le.fit_transform(fused["Type"])

baseline_features = ["Type_enc", "Air temperature [K]", "Process temperature [K]",
                      "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
enriched_features = baseline_features + new_features

X_base = fused[baseline_features]
X_enriched = fused[enriched_features]
y = fused["Machine failure"]

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

def cv_scores(X, y, cv):
    auc_scores, f1_scores_ = [], []
    for train_idx, test_idx in cv.split(X, y):
        model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        proba = model.predict_proba(X.iloc[test_idx])[:, 1]
        pred = model.predict(X.iloc[test_idx])
        auc_scores.append(roc_auc_score(y.iloc[test_idx], proba))
        f1_scores_.append(f1_score(y.iloc[test_idx], pred))
    return np.array(auc_scores), np.array(f1_scores_)

auc_base, f1_base = cv_scores(X_base, y, cv)
auc_enriched, f1_enriched = cv_scores(X_enriched, y, cv)

print(f"Baseline  -- AUC: {auc_base.mean():.4f} +/- {auc_base.std():.4f} | F1: {f1_base.mean():.4f} +/- {f1_base.std():.4f}")
print(f"Enriched  -- AUC: {auc_enriched.mean():.4f} +/- {auc_enriched.std():.4f} | F1: {f1_enriched.mean():.4f} +/- {f1_enriched.std():.4f}")

In [ ]:
def report_test(name, base_scores, enriched_scores):
    diff = enriched_scores - base_scores
    t_stat, p_t = stats.ttest_rel(enriched_scores, base_scores)
    w_stat, p_w = stats.wilcoxon(enriched_scores, base_scores)
    cohens_d = diff.mean() / diff.std(ddof=1)
    print(f"--- {name} ---")
    print(f"Mean improvement: {diff.mean():+.4f}")
    print(f"Paired t-test:    t={t_stat:.3f}, p={p_t:.4f}")
    print(f"Wilcoxon test:    W={w_stat:.3f}, p={p_w:.4f}")
    print(f"Cohen's d:        {cohens_d:.3f}\n")

report_test("ROC-AUC", auc_base, auc_enriched)
report_test("F1", f1_base, f1_enriched)

### Interpreting the Result

- If **p < 0.05** for the paired t-test/Wilcoxon test, you have statistical grounds to claim the external features provide a genuine, non-random improvement in predictive power.
- If p >= 0.05, report it honestly: it may mean the *simulated* external signal is too weakly correlated with `Machine failure` to matter — a valid and common finding worth a sentence on what a *stronger* real-world signal (true ambient sensor data) might add.
- Cohen's d gives the effect size (<0.2 negligible, 0.2-0.5 small, 0.5-0.8 medium, >0.8 large) — report it alongside the p-value, since with enough folds even tiny improvements can be statistically significant.

In [ ]:
model_enriched = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model_enriched.fit(X_enriched, y)

importances = pd.Series(model_enriched.feature_importances_, index=enriched_features).sort_values()

plt.figure(figsize=(8, 5))
colors = ["#d95f02" if f in new_features else "#1f78b4" for f in importances.index]
importances.plot(kind="barh", color=colors)
plt.title("Feature Importance -- Enriched Model\n(orange = new external/derived features)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
fused.to_csv("ai4i_fused_week2.csv", index=False)
data_dict.to_csv("data_dictionary_week2.csv", index=False)
print("Saved: ai4i_fused_week2.csv, data_dictionary_week2.csv")